# 🏥 TP — Transfer Learning pour la détection des EPI
## Fine-tuning de YOLOv8s (pré-entraîné sur COCO/ImageNet) → 3 classes EPI
### Dataset : Personal Protective Equipment - Combined Model (Roboflow Universe, CC BY 4.0)

---

**Objectifs pédagogiques :**
1. Charger un vrai modèle YOLOv8 pré-entraîné sur COCO/ImageNet
2. Ré-entraîner ses dernières couches sur des classes EPI (Phase 1)
3. Analyser les métriques : mAP@0.5, mAP@0.5:0.95, AP par classe, Précision, Recall
4. Décider si un fine-tuning profond (Phase 2) est justifié et le réaliser

**Classes détectées (3) — sous-ensemble PPE Combined Model v8 :**
| ID | Classe | Description |
|---|---|---|
| 0 | `Gloves` | Gants de protection portés |
| 1 | `Mask` | Masque de protection porté |
| 2 | `NO-Mask` | Visage sans masque |

---
> ⚙️ **Machine locale, CPU uniquement** — dataset réduit à 500/100/50 images pour le TP.

---
## Section 0 — Installation des dépendances
À exécuter une seule fois. Décommenter si les bibliothèques ne sont pas encore installées.

In [1]:
# Section 0 — Installation
# Décommenter les lignes ci-dessous si les bibliothèques ne sont pas installées

# !pip install ultralytics==8.3.0
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# !pip install matplotlib seaborn pandas pillow

# Vérification des versions installées
import ultralytics
import torch
import matplotlib
import pandas as pd

print(f'Ultralytics (YOLO) : {ultralytics.__version__}')
print(f'PyTorch            : {torch.__version__}')
print(f'Matplotlib         : {matplotlib.__version__}')
print(f'Pandas             : {pd.__version__}')
print(f'CPU disponible     : {True}')
print(f'GPU disponible     : {torch.cuda.is_available()} (non requis pour ce TP)')

Ultralytics (YOLO) : 8.3.0
PyTorch            : 2.12.0+cpu
Matplotlib         : 3.10.9
Pandas             : 3.0.3
CPU disponible     : True
GPU disponible     : False (non requis pour ce TP)


---
## Section 1 — Chargement du modèle pré-entraîné YOLOv8s

YOLOv8s a été pré-entraîné sur **COCO** (330K images, 80 classes incluant des personnes et vêtements)
et son backbone sur **ImageNet** (1,2M images). Ses poids encodent une connaissance générique des formes humaines,
textures de tissu et couleurs — directement réutilisable pour détecter des EPI.

Le fichier `yolov8s.pt` (~22 Mo) est téléchargé automatiquement depuis GitHub Ultralytics si absent.

In [2]:
# Section 1 — Chargement du modèle YOLOv8s pré-entraîné

from ultralytics import YOLO

# Téléchargement automatique si absent (depuis https://github.com/ultralytics/assets)
# Le modèle est entraîné sur COCO (80 classes) + backbone ImageNet
MODEL_PRETRAINED = 'yolov8s.pt'

model_base = YOLO(MODEL_PRETRAINED)

# Afficher les informations sur le modèle chargé
print('=' * 60)
print('MODÈLE PRÉ-ENTRAÎNÉ CHARGÉ : YOLOv8s')
print('=' * 60)
model_base.info(detailed=False)
print()
print('Nombre de classes COCO (avant fine-tuning) :', model_base.model.nc)
print('Dataset source                              :', 'COCO + ImageNet backbone')
print('Poids chargés depuis                        :', MODEL_PRETRAINED)

MODÈLE PRÉ-ENTRAÎNÉ CHARGÉ : YOLOv8s
YOLOv8s summary: 225 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs

Nombre de classes COCO (avant fine-tuning) : 80
Dataset source                              : COCO + ImageNet backbone
Poids chargés depuis                        : yolov8s.pt


---
## Section 2 — Préparation du dataset EPI

### Structure attendue
```
ppe_project/
├── dataset/
│   ├── images/
│   │   ├── train/    ← 80% des images
│   │   ├── val/      ← 15% des images
│   │   └── test/     ← 5% des images
│   └── labels/
│       ├── train/    ← annotations .txt (format YOLO)
│       ├── val/
│       └── test/
└── ppe_dataset.yaml
```

**Format d'une annotation YOLO** (fichier `.txt`) :  
`<classe_id> <x_centre> <y_centre> <largeur> <hauteur>`  
Toutes les valeurs sont normalisées entre 0 et 1 (relatives à la taille de l'image).

> 📦 La **Section 2b** (cellules suivantes) propose trois méthodes pour alimenter ce dataset.
> Exécuter l'**Option A** pour un démarrage immédiat.


In [3]:
# Section 2 — Configuration du dataset et création du fichier YAML

import os
import yaml
from pathlib import Path

# ── Paramètres du dataset ─────────────────────────────────────────────────────
PROJECT_DIR = Path('./ppe_project')
DATASET_DIR = PROJECT_DIR / 'dataset'

# Classes retenues : Mask, NO-Mask, Gloves (sous-ensemble du PPE Combined Model v8)
# Les annotations des autres classes seront ignorées lors de l'entraînement.
CLASS_NAMES = [
    'Gloves',  # 0
    'Mask',    # 1
    'NO-Mask', # 2
]
NUM_CLASSES = len(CLASS_NAMES)  # 3

# ── Créer la structure de dossiers ────────────────────────────────────────────
for split in ['train', 'val', 'test']:
    (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print('Structure de dossiers créée :')
for p in sorted(DATASET_DIR.rglob('*')):
    if p.is_dir():
        print(f'  {p}')

# ── Créer le fichier ppe_dataset.yaml ─────────────────────────────────────────
yaml_config = {
    'path': str(DATASET_DIR.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    NUM_CLASSES,
    'names': CLASS_NAMES,
}

YAML_PATH = PROJECT_DIR / 'ppe_dataset.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_config, f, default_flow_style=False, allow_unicode=True)

print(f'\nFichier de configuration créé : {YAML_PATH}')
print('\nContenu :')
with open(YAML_PATH) as f:
    print(f.read())

Structure de dossiers créée :
  ppe_project\dataset\images
  ppe_project\dataset\images\test
  ppe_project\dataset\images\train
  ppe_project\dataset\images\val
  ppe_project\dataset\labels
  ppe_project\dataset\labels\test
  ppe_project\dataset\labels\train
  ppe_project\dataset\labels\val

Fichier de configuration créé : ppe_project\ppe_dataset.yaml

Contenu :
names:
- mask_correct
- mask_incorrect
- gloves_correct
- gloves_absent
- gown_correct
- head_visible
nc: 6
path: C:\Users\Luc\Documents\50 MScAIB\53 MSc AIB Certificat-03 Cours & Data\53-02
  Méthodes actuelles du DL\TP CNN Détecteur_EPI_YOLOv8\ppe_project\dataset
test: images/test
train: images/train
val: images/val



In [4]:
# Section 2 (suite) — Vérification du dataset : comptage et distribution

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter

def count_annotations(labels_dir: Path):
    """Compte les annotations par classe dans un dossier de labels YOLO."""
    counter = Counter()
    label_files = list(labels_dir.glob('*.txt'))
    for label_file in label_files:
        with open(label_file) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    return counter, len(label_files)

# Vérifier le contenu du dataset pour chaque split
print('INVENTAIRE DU DATASET')
print('=' * 55)
total_imgs = 0
for split in ['train', 'val', 'test']:
    label_dir = DATASET_DIR / 'labels' / split
    img_dir   = DATASET_DIR / 'images' / split
    n_imgs = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
    n_labels = len(list(label_dir.glob('*.txt')))
    total_imgs += n_imgs
    print(f'  {split:<6}: {n_imgs:>4} images | {n_labels:>4} fichiers labels')
print(f'  TOTAL  : {total_imgs:>4} images')

print()
print('→ Placez vos images et labels dans les dossiers ci-dessus pour continuer.')
print('→ Conseil : Roboflow Universe propose des datasets EPI annotés au format YOLO.')

INVENTAIRE DU DATASET
  train :    0 images |    0 fichiers labels
  val   :    0 images |    0 fichiers labels
  test  :    0 images |    0 fichiers labels
  TOTAL  :    0 images

→ Placez vos images et labels dans les dossiers ci-dessus pour continuer.
→ Conseil : Roboflow Universe propose des datasets EPI annotés au format YOLO.


---
## Section 2b — Alimentation du dataset : trois options

La structure de dossiers est créée mais **vide**. Il faut maintenant y placer des images annotées.
Trois options sont disponibles selon vos ressources et le temps disponible.

| Option | Source | Temps | Annotation requise | Recommandé si... |
|--------|--------|-------|-------------------|------------------|
| **A** | Roboflow Universe (dataset public prêt) | 5 min | Non — déjà faite | TP court, découverte |
| **B** | Images libres + annotation manuelle LabelImg | 2–4h | Oui | Personnalisation totale |
| **C** | Capture webcam + annotation semi-auto | 1–2h | Partielle | Données propres à votre environnement |

> ⚡ **Pour ce TP : exécutez l'Option A** — dataset prêt en 5 minutes, entraînement immédiat.
> Les Options B et C sont documentées pour aller plus loin.

### Option A — Téléchargement d'un dataset public Roboflow (recommandé pour le TP)

In [36]:
# Option A — Téléchargement automatique d'un dataset EPI annoté depuis Roboflow Universe
# Dataset utilisé : 'Personal Protective Equipment - Combined Model' par Roboflow Universe Projects
# URL Roboflow : https://universe.roboflow.com/roboflow-universe-projects/personal-protective-equipment-combined-model
# Licence : CC BY 4.0 — 44 000 images, classes : Hardhat, Safety Vest, Mask, Gloves, Goggles, Person...

# ── Étape A1 : installer le client Roboflow ───────────────────────────────────
# !pip install roboflow  # Décommenter si pas encore installé

from roboflow import Roboflow
from pathlib import Path
import shutil, os, yaml

PROJECT_DIR  = Path('./ppe_project')
DATASET_DIR  = PROJECT_DIR / 'dataset'
DOWNLOAD_DIR = PROJECT_DIR / 'roboflow_download'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# ── Étape A2 : télécharger le dataset ────────────────────────────────────────
# Créez un compte gratuit sur roboflow.com pour obtenir votre clé personnelle
# Votre clé est disponible sur : https://app.roboflow.com/settings/api
ROBOFLOW_API_KEY = "rf_OBykoBmEHjYLrVgkoA9SNPFKZY72"  # Remplacer par votre clé

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Dataset PPE Combined Model — 44 000 images, version 8 (la plus récente)
project  = rf.workspace("roboflow-universe-projects").project("personal-protective-equipment-combined-model")
version  = project.version(8)

# Télécharge les images + annotations au format YOLOv8
dataset_rf = version.download(
    model_format="yolov8",       # Format compatible avec notre notebook
    location=str(DOWNLOAD_DIR),  # Dossier de destination temporaire
    overwrite=True
)

print(f'Dataset téléchargé dans : {DOWNLOAD_DIR}')
print(f'Chemin Roboflow : {dataset_rf.location}')

# ── Étape A3 : recopier dans notre structure ppe_project/dataset/ ─────────────
# Roboflow crée sa propre arborescence train/valid/test
# On la copie vers notre structure dataset/images/ et dataset/labels/

RF_SPLIT_MAP = {'train': 'train', 'valid': 'val', 'test': 'test'}
n_copied = {'images': 0, 'labels': 0}

for rf_split, our_split in RF_SPLIT_MAP.items():
    rf_img_dir    = DOWNLOAD_DIR / rf_split / 'images'
    rf_label_dir  = DOWNLOAD_DIR / rf_split / 'labels'
    dst_img_dir   = DATASET_DIR / 'images' / our_split
    dst_label_dir = DATASET_DIR / 'labels' / our_split
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_label_dir.mkdir(parents=True, exist_ok=True)

    if not rf_img_dir.exists():
        print(f'  Split {rf_split} absent — ignoré')
        continue

    # Copier les images
    for img in rf_img_dir.glob('*'):
        shutil.copy2(img, dst_img_dir / img.name)
        n_copied['images'] += 1

    # Copier les labels
    for lbl in rf_label_dir.glob('*.txt'):
        shutil.copy2(lbl, dst_label_dir / lbl.name)
        n_copied['labels'] += 1

print(f"\nCopié : {n_copied['images']} images, {n_copied['labels']} labels")
print('→ Exécuter la cellule suivante pour réduire le dataset et générer le YAML.')

loading Roboflow workspace...
loading Roboflow project...
<bound method Project.versions of <roboflow.core.project.Project object at 0x0000024E94FE5FD0>>


NameError: name 'version' is not defined

In [ ]:
# ── Étape A4 : Réduction du dataset pour un entraînement rapide sur CPU ───────
# Le dataset complet (~44 000 images) prendrait plusieurs jours sur CPU.
# On garde un sous-ensemble représentatif suffisant pour le TP.
#
# Temps estimé après réduction (CPU) :
#   Phase 1 (10 epochs, freeze=10) : ~30–60 min
#   Phase 2 (10 epochs, full)      : ~45–90 min

import random
from pathlib import Path
import yaml

PROJECT_DIR  = Path('./ppe_project')
DATASET_DIR  = PROJECT_DIR / 'dataset'
DOWNLOAD_DIR = PROJECT_DIR / 'roboflow_download'
YAML_PATH    = PROJECT_DIR / 'ppe_dataset.yaml'

# ── Paramètres de réduction (ajuster selon votre machine) ────────────────────
MAX_IMAGES = {'train': 500, 'val': 100, 'test': 50}
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print('Réduction du dataset pour entraînement CPU :')
print(f'  Cible : {MAX_IMAGES["train"]} train | {MAX_IMAGES["val"]} val | {MAX_IMAGES["test"]} test')
print()

for split, max_n in MAX_IMAGES.items():
    img_dir   = DATASET_DIR / 'images' / split
    label_dir = DATASET_DIR / 'labels' / split
    all_imgs  = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    n_before  = len(all_imgs)

    if n_before <= max_n:
        print(f'  {split:<6}: {n_before} images — déjà sous le seuil, rien à faire')
        continue

    # Tirer aléatoirement les images à SUPPRIMER
    to_remove = random.sample(all_imgs, n_before - max_n)
    for img in to_remove:
        img.unlink()  # Supprimer l'image
        lbl = label_dir / (img.stem + '.txt')
        if lbl.exists():
            lbl.unlink()  # Supprimer le label associé

    n_after = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    print(f'  {split:<6}: {n_before} → {n_after} images (supprimé {n_before - n_after})')


# ── Étape A5b : filtrer les annotations — garder uniquement Gloves, Mask, NO-Mask ──
# Dans le dataset original : Gloves=0, Mask=4, NO-Mask=6
# Après remappage         : Gloves=0, Mask=1, NO-Mask=2
KEEP_MAP = {0: 0, 4: 1, 6: 2}  # {ancien_id: nouvel_id}

print('\nFiltrage des annotations (conservation Gloves/Mask/NO-Mask uniquement) :')
n_kept = n_removed_ann = 0

for split in ['train', 'val', 'test']:
    lbl_dir = DATASET_DIR / 'labels' / split
    img_dir = DATASET_DIR / 'images' / split
    for lbl_file in list(lbl_dir.glob('*.txt')):
        new_lines = []
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                old_id = int(parts[0])
                if old_id in KEEP_MAP:
                    parts[0] = str(KEEP_MAP[old_id])
                    new_lines.append(' '.join(parts) + '\n')
                    n_kept += 1
                else:
                    n_removed_ann += 1
        if new_lines:
            with open(lbl_file, 'w') as f:
                f.writelines(new_lines)
        else:
            # Aucune annotation utile → supprimer image + label
            lbl_file.unlink()
            for ext in ['.jpg', '.jpeg', '.png']:
                img = img_dir / (lbl_file.stem + ext)
                if img.exists():
                    img.unlink()

print(f'  Annotations conservées : {n_kept}')
print(f'  Annotations supprimées : {n_removed_ann}')
# ── Étape A5 : lire les classes réelles du dataset et réécrire le YAML ────────
rf_yaml_candidates = list(DOWNLOAD_DIR.glob('*.yaml'))
if rf_yaml_candidates:
    with open(rf_yaml_candidates[0]) as f:
        rf_yaml = yaml.safe_load(f)
    CLASS_NAMES = rf_yaml.get('names', [])
    print(f'\nClasses détectées ({len(CLASS_NAMES)}) :', CLASS_NAMES)
else:
    # Fallback : classes retenues pour le TP
    CLASS_NAMES = ['Gloves', 'Mask', 'NO-Mask']
    print('YAML Roboflow non trouvé — classes fallback utilisées.')

# Réécrire le YAML avec les classes réelles
yaml_config = {
    'path':  str(DATASET_DIR.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    len(CLASS_NAMES),
    'names': CLASS_NAMES,
}
with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_config, f, default_flow_style=False, allow_unicode=True)

# ── Bilan final ───────────────────────────────────────────────────────────────
print()
print('=' * 55)
print('BILAN DATASET FINAL')
print('=' * 55)
total = 0
for split in ['train', 'val', 'test']:
    img_dir = DATASET_DIR / 'images' / split
    lbl_dir = DATASET_DIR / 'labels' / split
    n_img = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    n_lbl = len(list(lbl_dir.glob('*.txt')))
    total += n_img
    print(f'  {split:<6}: {n_img:>4} images | {n_lbl:>4} labels')
print(f'  TOTAL  : {total:>4} images')
print(f'  Classes: {len(CLASS_NAMES)}')
print(f'  YAML   : {YAML_PATH}')
print('=' * 55)
print('\nPrêt pour l\'entraînement → passer à la Section 3')

In [ ]:
# Option A (alternative) — Téléchargement direct par URL publique sans compte Roboflow
# Utilise un dataset EPI public sur GitHub (format YOLOv8, licence MIT)
# Dataset : 'EfficientDet-PPE' adapté YOLOv8, ~600 images annotées
#
# Alternative si l'API Roboflow n'est pas accessible :
# Télécharger manuellement depuis l'un de ces dépôts GitHub :
#   https://github.com/nickvdyck/weee-detection (EPI industriels, 80% similaires)
#   https://universe.roboflow.com  → chercher 'hospital PPE' → Export YOLOv8

import urllib.request, zipfile, shutil
from pathlib import Path
import yaml

PROJECT_DIR = Path('./ppe_project')
DATASET_DIR = PROJECT_DIR / 'dataset'
YAML_PATH   = PROJECT_DIR / 'ppe_dataset.yaml'
ZIP_PATH    = PROJECT_DIR / 'ppe_dataset.zip'

# ── URL du dataset public (format YOLOv8, ~600 images EPI) ───────────────────
# Remplacer par l'URL de votre choix si vous avez téléchargé manuellement
DATASET_URL = (
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
    "coco8.zip"  # Dataset de démo Ultralytics — 8 images pour tester le pipeline
    # Pour un vrai dataset EPI, remplacer par l'URL de l'export Roboflow
)

print(f'Téléchargement depuis : {DATASET_URL}')
urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
print(f'Téléchargé : {ZIP_PATH.stat().st_size / 1024:.0f} Ko')

# ── Décompression ─────────────────────────────────────────────────────────────
EXTRACT_DIR = PROJECT_DIR / 'zip_extract'
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print(f'Extrait dans : {EXTRACT_DIR}')

# Afficher l'arborescence extraite pour comprendre la structure
print('\nContenu extrait :')
for p in sorted(EXTRACT_DIR.rglob('*'))[:30]:  # Limité à 30 lignes
    indent = '  ' * (len(p.parts) - len(EXTRACT_DIR.parts))
    print(f'{indent}{p.name}{chr(47) if p.is_dir() else ""}')

print('\n→ Adapter le chemin source ci-dessous selon la structure affichée')
print('→ Puis exécuter la cellule de copie de l\'Option A (Étape A3)')

### Option B — Images propres + annotation manuelle avec LabelImg

In [ ]:
# Option B — Constituer son propre dataset avec des images libres de droits
# et les annoter manuellement avec LabelImg (outil gratuit, interface graphique).
#
# ÉTAPES :
#   1. Collecter des images (sources proposées ci-dessous)
#   2. Installer et lancer LabelImg
#   3. Annoter les objets EPI dans chaque image
#   4. Exporter au format YOLO → fichiers .txt générés automatiquement
#   5. Répartir train/val/test avec la cellule de split ci-dessous

# ── Étape B1 : installer LabelImg ─────────────────────────────────────────────
# !pip install labelImg  # Décommenter et exécuter une fois

# ── Étape B2 : sources d'images libres de droits ──────────────────────────────
SOURCES_IMAGES = {
    'Unsplash':   'https://unsplash.com/s/photos/hospital-worker',
    'Pixabay':    'https://pixabay.com/images/search/medical+mask/',
    'Pexels':     'https://www.pexels.com/search/nurse%20mask/',
    'NLM':        'https://www.nlm.nih.gov/ocal/index.html',  # images médicales US
}
print('Sources d\'images libres de droits pour les EPI hospitaliers :')
for nom, url in SOURCES_IMAGES.items():
    print(f'  {nom:<12} → {url}')

# ── Étape B3 : lancer LabelImg ────────────────────────────────────────────────
# Dans un terminal (PAS dans Jupyter), exécuter :
#   labelImg
#
# Dans LabelImg :
#   1. Open Dir      → sélectionner votre dossier d'images brutes
#   2. Change Save Dir → sélectionner un dossier de sortie pour les .txt
#   3. PascalVOC     → changer en YOLO (menu en bas à gauche)
#   4. Annoter chaque image : W = nouveau rectangle, A/D = image précédente/suivante
#   5. Ctrl+S = sauvegarder l'annotation courante

import subprocess, sys

print('\nPour lancer LabelImg depuis ce notebook :')
print('  Exécuter la ligne ci-dessous dans une cellule (décommenter)')
print()
print('  # import subprocess')
print('  # subprocess.Popen([sys.executable, "-m", "labelImg"])')

# ── Étape B4 : vérifier que les fichiers .txt sont au format YOLO ─────────────
# LabelImg en mode YOLO génère des fichiers .txt avec :
#   <id_classe> <x_centre> <y_centre> <largeur> <hauteur>
# et un fichier classes.txt listant les noms dans l'ordre

from pathlib import Path

RAW_LABELS_DIR = Path('./mes_annotations')  # Dossier où LabelImg a sauvegardé les .txt

if RAW_LABELS_DIR.exists():
    txt_files = list(RAW_LABELS_DIR.glob('*.txt'))
    print(f'\nFichiers .txt trouvés : {len(txt_files)}')
    if txt_files:
        sample = txt_files[0]
        print(f'Exemple ({sample.name}) :')
        with open(sample) as f:
            for line in f:
                print(f'  {line.rstrip()}')
else:
    print(f'Dossier {RAW_LABELS_DIR} non trouvé.')
    print('→ Créer ce dossier et y placer les fichiers .txt générés par LabelImg.')

In [ ]:
# Option B — Répartition automatique train/val/test
# À exécuter après avoir annoté toutes les images avec LabelImg
# et placé les images dans RAW_IMAGES_DIR et les labels dans RAW_LABELS_DIR

import shutil, random
from pathlib import Path

PROJECT_DIR   = Path('./ppe_project')
DATASET_DIR   = PROJECT_DIR / 'dataset'

RAW_IMAGES_DIR = Path('./mes_images')      # Dossier contenant toutes les images brutes
RAW_LABELS_DIR = Path('./mes_annotations') # Dossier contenant tous les .txt LabelImg

# ── Paramètres de split ───────────────────────────────────────────────────────
TRAIN_RATIO = 0.80  # 80% pour l'entraînement
VAL_RATIO   = 0.15  # 15% pour la validation
TEST_RATIO  = 0.05  # 5%  pour le test final
RANDOM_SEED = 42    # Graine aléatoire pour reproductibilité

# ── Lister les paires image/label valides ─────────────────────────────────────
random.seed(RANDOM_SEED)

valid_pairs = []
for ext in ['*.jpg', '*.jpeg', '*.png']:
    for img_path in RAW_IMAGES_DIR.glob(ext):
        label_path = RAW_LABELS_DIR / (img_path.stem + '.txt')
        if label_path.exists():
            valid_pairs.append((img_path, label_path))
        else:
            print(f'  ⚠ Label manquant pour : {img_path.name}')

print(f'Paires image/label valides : {len(valid_pairs)}')
if len(valid_pairs) < 20:
    print('⚠  Moins de 20 paires — résultats d\'entraînement non représentatifs.')
    print('   Recommandé : minimum 200 images pour une détection fiable.')

# ── Mélange aléatoire et découpage ───────────────────────────────────────────
random.shuffle(valid_pairs)
n = len(valid_pairs)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

splits = {
    'train': valid_pairs[:n_train],
    'val':   valid_pairs[n_train:n_train + n_val],
    'test':  valid_pairs[n_train + n_val:],
}

# ── Copier dans dataset/ ──────────────────────────────────────────────────────
for split_name, pairs in splits.items():
    dst_img = DATASET_DIR / 'images' / split_name
    dst_lbl = DATASET_DIR / 'labels' / split_name
    for img_path, lbl_path in pairs:
        shutil.copy2(img_path, dst_img / img_path.name)
        shutil.copy2(lbl_path, dst_lbl / lbl_path.name)
    print(f'  {split_name:<6} : {len(pairs):>4} paires copiées')

print(f'\nDataset prêt dans : {DATASET_DIR}')
print('→ Passer à la Section 3 pour lancer l\'entraînement')

### Option C — Capture webcam + annotation semi-automatique (Grounding DINO)

In [ ]:
# Option C — Capturer des images avec la webcam et les annoter automatiquement
# Idéal pour produire des données propres à votre environnement (éclairage, angles).
#
# Pipeline :
#   1. Capturer N images avec la webcam (porteurs d'EPI filmés)
#   2. Annoter automatiquement avec un modèle zero-shot (Grounding DINO)
#      qui génère des boîtes englobantes à partir de descriptions textuelles
#   3. Vérifier visuellement et corriger les erreurs (10-20% des images)
#   4. Répartir train/val/test avec le script de l'Option B

# !pip install groundingdino-py opencv-python Pillow  # Décommenter si absent

import cv2
from pathlib import Path
from datetime import datetime

PROJECT_DIR   = Path('./ppe_project')
RAW_IMAGES_DIR = PROJECT_DIR / 'raw_webcam'
RAW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# ── Étape C1 : capture webcam ─────────────────────────────────────────────────
N_FRAMES    = 50   # Nombre d'images à capturer
INTERVAL_S  = 2    # Intervalle entre captures (secondes)
CAM_INDEX   = 0    # Index de la caméra (0 = webcam intégrée)

cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    print('⚠  Webcam non disponible. Vérifier CAM_INDEX ou utiliser les Options A/B.')
else:
    import time
    n_saved = 0
    print(f'Capture de {N_FRAMES} images (appuyer sur Q pour arrêter)...')
    while n_saved < N_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break
        ts   = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        path = RAW_IMAGES_DIR / f'capture_{ts}.jpg'
        cv2.imwrite(str(path), frame)
        n_saved += 1
        print(f'  Sauvegardé : {path.name}  ({n_saved}/{N_FRAMES})', end='\r')
        time.sleep(INTERVAL_S)
    cap.release()
    print(f'\n{n_saved} images capturées dans : {RAW_IMAGES_DIR}')

# ── Étape C2 : annotation zero-shot avec Grounding DINO ──────────────────────
# Grounding DINO génère des boîtes englobantes à partir de descriptions textuelles.
# Il n'a PAS besoin d'être entraîné sur EPI — il comprend le langage naturel.

PROMPTS_EPI = [
    'gloves on hands',            # → classe 0 Gloves
    'safety goggles glasses',     # → classe 1 Goggles
    'hard hat helmet',            # → classe 2 Hardhat
    'no hard hat helmet',         # → classe 5 NO-Hardhat
    'no face mask',               # → classe 6 NO-Mask
    'person worker',              # → classe 8 Person
    'safety vest jacket',         # → classe 10 Safety Vest
]

PROMPT_TO_CLASS = {
    'gloves on hands':            0,
    'safety goggles glasses':     1,
    'hard hat helmet':            2,
    'no hard hat helmet':         5,
    'no face mask':               6,
    'person worker':              8,
    'safety vest jacket':         10,
}

print('\nAnnotation semi-automatique avec Grounding DINO :')
print('Prompts utilisés :')
for prompt, cls_id in PROMPT_TO_CLASS.items():
    CLASS_NAMES = ['Gloves', 'Mask', 'NO-Mask']
    print(f'  [{cls_id}] {CLASS_NAMES[cls_id]:<20} ← "{prompt}"')

print('\n→ Exécuter la cellule suivante pour lancer l\'annotation automatique')

In [ ]:
# Option C — Annotation automatique des images capturées par Grounding DINO
# Produit des fichiers .txt au format YOLO pour chaque image

# !pip install groundingdino-py  # Décommenter si absent

from pathlib import Path
from PIL import Image
import torch

PROJECT_DIR    = Path('./ppe_project')
RAW_IMAGES_DIR = PROJECT_DIR / 'raw_webcam'
AUTO_LABELS_DIR= PROJECT_DIR / 'auto_labels'
AUTO_LABELS_DIR.mkdir(parents=True, exist_ok=True)

# Seuil de confiance : boîtes sous ce seuil sont ignorées
CONF_THRESHOLD = 0.35

CLASS_NAMES = ['Gloves', 'Mask', 'NO-Mask']

PROMPTS_EPI = [
    ('gloves on hands',            0),  # Gloves
    ('safety goggles glasses',     1),  # Goggles
    ('hard hat helmet',            2),  # Hardhat
    ('no hard hat helmet',         5),  # NO-Hardhat
    ('no face mask',               6),  # NO-Mask
    ('person worker',              8),  # Person
    ('safety vest jacket',         10), # Safety Vest
]

try:
    from groundingdino.util.inference import load_model, load_image, predict
    import groundingdino.datasets.transforms as T

    # Charger le modèle Grounding DINO (téléchargement automatique ~680 Mo)
    model = load_model(
        'groundingdino/config/GroundingDINO_SwinT_OGC.py',
        'weights/groundingdino_swint_ogc.pth'
    )

    images = list(RAW_IMAGES_DIR.glob('*.jpg')) + list(RAW_IMAGES_DIR.glob('*.png'))
    print(f'Images à annoter : {len(images)}')

    n_annotated = 0
    for img_path in images:
        img_pil = Image.open(img_path).convert('RGB')
        W, H    = img_pil.size
        image_source, image_tensor = load_image(str(img_path))

        annotations = []  # Lignes YOLO pour cette image

        for prompt_text, class_id in PROMPTS_EPI:
            boxes, logits, phrases = predict(
                model=model,
                image=image_tensor,
                caption=prompt_text,
                box_threshold=CONF_THRESHOLD,
                text_threshold=CONF_THRESHOLD,
            )
            # boxes = tenseur Nx4 en coordonnées normalisées [cx,cy,w,h]
            for box in boxes:
                cx, cy, bw, bh = box.tolist()
                annotations.append(f'{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

        # Écrire le fichier label
        label_path = AUTO_LABELS_DIR / (img_path.stem + '.txt')
        with open(label_path, 'w') as f:
            f.write('\n'.join(annotations))
        n_annotated += 1
        print(f'  Annoté : {img_path.name}  ({len(annotations)} objets)  ({n_annotated}/{len(images)})', end='\r')

    print(f'\n{n_annotated} images annotées → {AUTO_LABELS_DIR}')
    print('→ Vérifier visuellement 10-20% des annotations, puis exécuter le split (Option B)')

except ImportError:
    print('Grounding DINO non installé.')
    print('Installer avec : pip install groundingdino-py')
    print('Ou utiliser LabelImg manuellement (Option B).')

In [ ]:
# Vérification finale du dataset — commune aux Options A, B et C
# À exécuter après avoir alimenté le dataset (quelle que soit l'option choisie)

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import Counter

PROJECT_DIR = Path('./ppe_project')
DATASET_DIR = PROJECT_DIR / 'dataset'

CLASS_NAMES = ['Gloves', 'Mask', 'NO-Mask']

print('═' * 60)
print('  BILAN DU DATASET')
print('═' * 60)

total_annotations = Counter()
ok_to_train = True

for split in ['train', 'val', 'test']:
    img_dir = DATASET_DIR / 'images' / split
    lbl_dir = DATASET_DIR / 'labels' / split

    imgs   = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    labels = list(lbl_dir.glob('*.txt'))
    n_img, n_lbl = len(imgs), len(labels)

    # Compter les annotations par classe
    class_counter = Counter()
    for lbl in labels:
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_counter[int(parts[0])] += 1
    total_annotations += class_counter

    status = '✔' if n_img == n_lbl and n_img > 0 else '✖'
    print(f'  {status} {split:<6} : {n_img:>4} images | {n_lbl:>4} labels')
    if n_img != n_lbl:
        print(f'    ⚠  Désynchronisation : {abs(n_img-n_lbl)} fichiers sans correspondant')
        ok_to_train = False
    if n_img == 0:
        ok_to_train = False

print()
print('  Annotations par classe (train+val+test) :')
for cls_id, cls_name in enumerate(CLASS_NAMES):
    count = total_annotations.get(cls_id, 0)
    barre = '█' * min(count // 10, 30)
    warn  = ' ⚠ FAIBLE' if count < 50 else ''
    print(f'  [{cls_id}] {cls_name:<20} {count:>5} annotations  {barre}{warn}')

print()
print('═' * 60)
if ok_to_train:
    print('  ✔ Dataset valide — passer à la Section 3 pour l\'entraînement')
else:
    print('  ✖ Problèmes détectés — corriger avant de lancer l\'entraînement')
print('═' * 60)

# ── Graphique de distribution des classes ─────────────────────────────────────
if total_annotations:
    counts = [total_annotations.get(i, 0) for i in range(len(CLASS_NAMES))]
    colors_bar = ['#c0392b' if c < 50 else '#2e75b6' for c in counts]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(CLASS_NAMES, counts, color=colors_bar, edgecolor='grey', linewidth=0.5)
    ax.axvline(x=50, color='red', linestyle='--', linewidth=1, label='Seuil minimum (50)')
    ax.set_xlabel('Nombre d\'annotations')
    ax.set_title('Distribution des annotations par classe', fontweight='bold')
    ax.legend(fontsize=9)
    for bar, val in zip(bars, counts):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(str(PROJECT_DIR / 'dataset_distribution.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Graphique sauvegardé : dataset_distribution.png')

---
## Section 3 — Phase 1 : Entraînement des dernières couches (backbone gelé)

### Principe du gel (`freeze`)
On **gèle** les 10 premières couches du backbone (paramètre `freeze=10`).  
Leurs poids **ne sont pas modifiés** pendant l'entraînement.  
Seule la **tête de détection** (detection head) apprend les 6 nouvelles classes EPI.

```
YOLOv8s
├── Backbone (couches 0-9)  ← GELÉ  — réutilise les connaissances COCO/ImageNet
├── Neck     (couches 10-21) ← GELÉ
└── Head     (couches 22+)  ← ENTRAÎNÉ — apprend les 6 classes EPI
```

**Hyperparamètres clés :**
- `lr0=0.01` — taux d'apprentissage initial
- `batch=8`  — réduire à 4 si RAM < 8 Go
- `epochs=50` — 50 passages sur toutes les images

In [ ]:
# Section 3 — Phase 1 : Fine-tuning avec backbone gelé (freeze=10)

from ultralytics import YOLO
from pathlib import Path

YAML_PATH   = Path('./ppe_project/ppe_dataset.yaml')
RUNS_DIR    = Path('./ppe_project/runs')

# ── Rechargement du modèle pré-entraîné ──────────────────────────────────────
model_p1 = YOLO('yolov8s.pt')

print('Phase 1 — Entraînement avec backbone gelé (freeze=10)')
print('Seule la tête de détection sera mise à jour.')
print('=' * 60)

# ── Lancement de l'entraînement Phase 1 ──────────────────────────────────────
results_p1 = model_p1.train(
    data    = str(YAML_PATH),
    epochs  = 50,          # Nombre de passages sur le dataset
    imgsz   = 640,         # Taille des images (pixels)
    batch   = 8,           # Images par lot (réduire à 4 si RAM insuffisante)
    lr0     = 0.01,        # Taux d'apprentissage initial
    lrf     = 0.01,        # Taux final = lr0 * lrf
    freeze  = 10,          # Geler les 10 premières couches du backbone
    device  = 'cpu',       # CPU — pas de GPU requis
    project = str(RUNS_DIR),
    name    = 'phase1_freeze10',
    save    = True,        # Sauvegarder les meilleurs poids (best.pt)
    plots   = True,        # Générer les courbes de loss et métriques
    verbose = True,
)

# ── Résumé Phase 1 ────────────────────────────────────────────────────────────
map50_p1    = results_p1.results_dict.get('metrics/mAP50(B)',    0)
map5095_p1  = results_p1.results_dict.get('metrics/mAP50-95(B)', 0)
prec_p1     = results_p1.results_dict.get('metrics/precision(B)', 0)
rec_p1      = results_p1.results_dict.get('metrics/recall(B)',    0)

print('\n' + '=' * 60)
print('RÉSULTATS PHASE 1 — BACKBONE GELÉ')
print('=' * 60)
print(f'  mAP@0.5        : {map50_p1:.4f}')
print(f'  mAP@0.5:0.95   : {map5095_p1:.4f}')
print(f'  Précision      : {prec_p1:.4f}')
print(f'  Recall         : {rec_p1:.4f}')

BEST_P1 = RUNS_DIR / 'phase1_freeze10' / 'weights' / 'best.pt'
print(f'\nMeilleurs poids sauvegardés : {BEST_P1}')

---
## Section 4 — Analyse des métriques de Phase 1

### Métriques expliquées

| Métrique | Formule | Ce qu'elle mesure |
|---|---|---|
| **Précision** | VP / (VP + FP) | Fiabilité des détections — peu de fausses alarmes |
| **Recall** | VP / (VP + FN) | Exhaustivité — peu d'objets manqués |
| **AP@0.5** | Aire sous la courbe P/R à IoU≥0.5 | Qualité globale par classe |
| **mAP@0.5** | Moyenne des AP@0.5 sur toutes les classes | Métrique principale |
| **mAP@0.5:0.95** | Moyenne sur 10 seuils IoU (50%→95%) | Métrique stricte (boîtes précises) |

**IoU (Intersection over Union)** = Surface d'intersection / Surface d'union entre boîte prédite et boîte réelle.

In [ ]:
# Section 4 — Évaluation détaillée Phase 1 : métriques par classe

from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

YAML_PATH = Path('./ppe_project/ppe_dataset.yaml')
BEST_P1   = Path('./ppe_project/runs/phase1_freeze10/weights/best.pt')

CLASS_NAMES = ['Gloves', 'Mask', 'NO-Mask']

model_eval_p1 = YOLO(str(BEST_P1))

# Évaluation sur le jeu de test
metrics_p1 = model_eval_p1.val(
    data  = str(YAML_PATH),
    split = 'test',
    iou   = 0.5,
    conf  = 0.25,
    verbose = True,
)

# ── Affichage tabulaire des métriques par classe ───────────────────────────────
ap50_per_class = metrics_p1.box.ap50      # AP@0.5 par classe
prec_per_class = metrics_p1.box.p         # Precision par classe
rec_per_class  = metrics_p1.box.r         # Recall par classe

print('\n' + '=' * 72)
print(f'{"CLASSE":<22} {"AP@0.5":>8} {"Précision":>10} {"Recall":>8}  Barre AP')
print('=' * 72)
for i, name in enumerate(CLASS_NAMES):
    ap   = ap50_per_class[i] if i < len(ap50_per_class) else 0
    prec = prec_per_class[i] if i < len(prec_per_class) else 0
    rec  = rec_per_class[i]  if i < len(rec_per_class)  else 0
    bar  = '█' * int(ap * 25)
    print(f'  {name:<20} {ap:>8.3f} {prec:>10.3f} {rec:>8.3f}  {bar}')

print('=' * 72)
print(f'  {"GLOBAL (mAP)":<20} {metrics_p1.box.map50:>8.3f} {metrics_p1.box.mp:>10.3f} {metrics_p1.box.mr:>8.3f}')
print(f'  mAP@0.5:0.95 = {metrics_p1.box.map:.4f}')

# ── Graphique : AP@0.5 par classe ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase 1 — Performances par classe (backbone gelé, freeze=10)', fontsize=13, fontweight='bold')

colors = plt.cm.RdYlGn([ap for ap in ap50_per_class])
bars = axes[0].barh(CLASS_NAMES, ap50_per_class, color=colors, edgecolor='grey', linewidth=0.5)
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=1, label='Seuil 0.50')
axes[0].axvline(x=0.7, color='orange', linestyle='--', linewidth=1, label='Seuil 0.70')
axes[0].set_xlabel('AP@0.5')
axes[0].set_title('AP@0.5 par classe')
axes[0].set_xlim(0, 1)
axes[0].legend(fontsize=9)
for bar, val in zip(bars, ap50_per_class):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

x = np.arange(len(CLASS_NAMES))
width = 0.35
axes[1].bar(x - width/2, prec_per_class, width, label='Précision', color='steelblue', alpha=0.8)
axes[1].bar(x + width/2, rec_per_class,  width, label='Recall',    color='coral',     alpha=0.8)
axes[1].axhline(y=0.7, color='gray', linestyle=':', linewidth=1)
axes[1].set_xticks(x)
axes[1].set_xticklabels([n.replace('_', '\n') for n in CLASS_NAMES], fontsize=8)
axes[1].set_ylabel('Score')
axes[1].set_title('Précision vs Recall par classe')
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()
plt.savefig('./ppe_project/phase1_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé : ./ppe_project/phase1_metrics.png')

In [ ]:
# Section 4 (suite) — Lecture et affichage des courbes d'entraînement Phase 1

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_CSV = Path('./ppe_project/runs/phase1_freeze10/results.csv')

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    df.columns = df.columns.str.strip()  # Nettoyer les espaces

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Phase 1 — Courbes d\'entraînement (backbone gelé)', fontsize=13, fontweight='bold')

    def plot_metric(ax, col, title, color='steelblue', ylabel=''):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=1.5)
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('Epoch')
            ax.set_ylabel(ylabel or col)
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'Colonne\n{col}\nnon trouvée', ha='center', va='center')
            ax.set_title(title)

    plot_metric(axes[0,0], 'train/box_loss',        'Loss boîtes (train)',    'steelblue', 'box_loss')
    plot_metric(axes[0,1], 'train/cls_loss',        'Loss classes (train)',   'coral',     'cls_loss')
    plot_metric(axes[0,2], 'train/dfl_loss',        'Loss DFL (train)',       'purple',    'dfl_loss')
    plot_metric(axes[1,0], 'val/box_loss',          'Loss boîtes (val)',      'navy',      'box_loss')
    plot_metric(axes[1,1], 'metrics/mAP50(B)',      'mAP@0.5',                'green',     'mAP@0.5')
    plot_metric(axes[1,2], 'metrics/mAP50-95(B)',   'mAP@0.5:0.95',          'darkgreen', 'mAP@0.5:0.95')

    plt.tight_layout()
    plt.savefig('./ppe_project/phase1_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Courbes sauvegardées : ./ppe_project/phase1_curves.png')
else:
    print(f'Fichier results.csv non trouvé : {RESULTS_CSV}')
    print('→ Assurez-vous que la Phase 1 a bien été exécutée.')

---
## Section 5 — Décision : faut-il réentraîner plus de couches ?

### Grille de décision

| Critère | Seuil Phase 2 justifiée | Analyse |
|---|---|---|
| mAP@0.5 global | < 0.70 | Insuffisant pour usage opérationnel |
| AP classe critique | < 0.50 | Classe trop difficile pour le head seul |
| Recall classe sécurité | < 0.60 | Risque de manquer des EPI absents |
| Overfitting (val loss ↑) | Si présent | Arrêter — Phase 2 aggraverait le surapprentissage |
| Dataset trop petit | < 200 images train | Phase 2 → surapprentissage garanti |

> **Règle générale :** si mAP@0.5 Phase 1 < 0.70, la Phase 2 est justifiée.  
> Si mAP@0.5 Phase 1 ≥ 0.75, les résultats sont suffisants — Phase 2 optionnelle.

In [ ]:
# Section 5 — Analyse automatique et décision Phase 2

from ultralytics import YOLO
from pathlib import Path

YAML_PATH = Path('./ppe_project/ppe_dataset.yaml')
BEST_P1   = Path('./ppe_project/runs/phase1_freeze10/weights/best.pt')
CLASS_NAMES = ['Gloves', 'Mask', 'NO-Mask']

model_dec = YOLO(str(BEST_P1))
metrics   = model_dec.val(data=str(YAML_PATH), split='test', iou=0.5, conf=0.25, verbose=False)

map50     = metrics.box.map50
map5095   = metrics.box.map
ap50_cls  = metrics.box.ap50
rec_cls   = metrics.box.r

# ── Critères de décision ──────────────────────────────────────────────────────
criteres = {
    'mAP@0.5 < 0.70'           : map50 < 0.70,
    'Écart mAP50 - mAP5095 > 0.20' : (map50 - map5095) > 0.20,
    'AP_min classe < 0.50'     : (min(ap50_cls) < 0.50) if len(ap50_cls) > 0 else False,
    'Recall_min < 0.60'        : (min(rec_cls) < 0.60)  if len(rec_cls) > 0  else False,
}

n_ok = sum(criteres.values())

print('ANALYSE DE DÉCISION — PHASE 2')
print('=' * 55)
print(f'  mAP@0.5        : {map50:.4f}')
print(f'  mAP@0.5:0.95   : {map5095:.4f}')
if len(ap50_cls): print(f'  AP min (classe) : {min(ap50_cls):.4f}  ({CLASS_NAMES[ap50_cls.argmin()]})')
if len(rec_cls):  print(f'  Recall min      : {min(rec_cls):.4f}  ({CLASS_NAMES[rec_cls.argmin()]})')
print()
print('Critères déclencheurs de la Phase 2 :')
for critere, atteint in criteres.items():
    statut = '✔ DÉCLENCHÉ' if atteint else '✖ ok'
    print(f'  [{statut}]  {critere}')

print()
print('─' * 55)
if n_ok >= 2:
    print(f'  → DÉCISION : PHASE 2 RECOMMANDÉE ({n_ok}/{len(criteres)} critères déclenchés)')
    print('    Le fine-tuning profond devrait améliorer significativement les résultats.')
    DO_PHASE2 = True
elif n_ok == 1:
    print(f'  → DÉCISION : PHASE 2 OPTIONNELLE ({n_ok}/{len(criteres)} critère déclenché)')
    print('    Amélioration possible mais les résultats sont déjà acceptables.')
    DO_PHASE2 = False
else:
    print(f'  → DÉCISION : PHASE 2 NON NÉCESSAIRE (0/{len(criteres)} critère déclenché)')
    print('    Les résultats de Phase 1 sont suffisants.')
    DO_PHASE2 = False
print('─' * 55)

---
## Section 6 — Phase 2 : Fine-tuning profond (toutes les couches)

En Phase 2, **aucune couche n'est gelée** (`freeze` n'est pas passé).  
Toutes les couches, y compris le backbone, s'adaptent aux images hospitalières.

⚠️ **Point critique :** Le taux d'apprentissage (`lr0`) doit être **10x plus faible** qu'en Phase 1.  
Un LR trop élevé détruirait les poids COCO/ImageNet appris — c'est le **catastrophic forgetting**.

```
Phase 1 : lr0=0.01,  freeze=10  → Head apprend, Backbone stable
Phase 2 : lr0=0.001, freeze=0   → Tout s'adapte doucement
```

In [ ]:
# Section 6 — Phase 2 : Fine-tuning toutes couches dégelées
# Exécuter seulement si DO_PHASE2 est True (décision Section 5)

from ultralytics import YOLO
from pathlib import Path

YAML_PATH = Path('./ppe_project/ppe_dataset.yaml')
BEST_P1   = Path('./ppe_project/runs/phase1_freeze10/weights/best.pt')
RUNS_DIR  = Path('./ppe_project/runs')

if not DO_PHASE2:
    print('Phase 2 non déclenchée par l\'analyse (Section 5).')
    print('Modifiez DO_PHASE2 = True pour l\'exécuter manuellement.')
else:
    print('Phase 2 — Fine-tuning profond : toutes les couches dégelées')
    print('Taux d\'apprentissage réduit (lr0=0.001) pour éviter le catastrophic forgetting')
    print('=' * 60)

    # Repartir des meilleurs poids Phase 1
    model_p2 = YOLO(str(BEST_P1))

    results_p2 = model_p2.train(
        data          = str(YAML_PATH),
        epochs        = 50,            # Epochs supplémentaires
        imgsz         = 640,
        batch         = 8,
        lr0           = 0.001,         # LR 10x plus faible qu'en Phase 1
        lrf           = 0.01,
        warmup_epochs = 3,             # Montée en puissance progressive du LR
        weight_decay  = 0.0005,        # Régularisation pour limiter l'overfitting
        # freeze non passé → toutes les couches sont dégelées
        device  = 'cpu',
        project = str(RUNS_DIR),
        name    = 'phase2_full_finetune',
        save    = True,
        plots   = True,
        verbose = True,
    )

    map50_p2   = results_p2.results_dict.get('metrics/mAP50(B)',    0)
    map5095_p2 = results_p2.results_dict.get('metrics/mAP50-95(B)', 0)
    prec_p2    = results_p2.results_dict.get('metrics/precision(B)', 0)
    rec_p2     = results_p2.results_dict.get('metrics/recall(B)',    0)

    print('\n' + '=' * 60)
    print('RÉSULTATS PHASE 2 — TOUTES COUCHES')
    print('=' * 60)
    print(f'  mAP@0.5        : {map50_p2:.4f}')
    print(f'  mAP@0.5:0.95   : {map5095_p2:.4f}')
    print(f'  Précision      : {prec_p2:.4f}')
    print(f'  Recall         : {rec_p2:.4f}')

    BEST_P2 = RUNS_DIR / 'phase2_full_finetune' / 'weights' / 'best.pt'
    print(f'\nMeilleurs poids sauvegardés : {BEST_P2}')

---
## Section 7 — Comparaison Phase 1 vs Phase 2 et conclusion

Cette section compare les performances des deux phases et produit un rapport visuel.

In [ ]:
# Section 7 — Comparaison complète Phase 1 vs Phase 2

from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

YAML_PATH   = Path('./ppe_project/ppe_dataset.yaml')
BEST_P1     = Path('./ppe_project/runs/phase1_freeze10/weights/best.pt')
BEST_P2     = Path('./ppe_project/runs/phase2_full_finetune/weights/best.pt')
CLASS_NAMES = ['Gloves', 'Mask', 'NO-Mask']

# Évaluation des deux modèles
print('Évaluation Phase 1...')
m1 = YOLO(str(BEST_P1)).val(data=str(YAML_PATH), split='test', iou=0.5, conf=0.25, verbose=False)

results_p2_available = BEST_P2.exists()
if results_p2_available:
    print('Évaluation Phase 2...')
    m2 = YOLO(str(BEST_P2)).val(data=str(YAML_PATH), split='test', iou=0.5, conf=0.25, verbose=False)

# ── Tableau comparatif ────────────────────────────────────────────────────────
print('\n' + '=' * 75)
print(f'{"CLASSE":<22} {"AP@0.5 P1":>10} ', end='')
if results_p2_available:
    print(f'{"AP@0.5 P2":>10} {"Gain":>8}')
else:
    print()
print('=' * 75)

for i, name in enumerate(CLASS_NAMES):
    ap1 = m1.box.ap50[i] if i < len(m1.box.ap50) else 0
    print(f'  {name:<20} {ap1:>10.3f} ', end='')
    if results_p2_available:
        ap2  = m2.box.ap50[i] if i < len(m2.box.ap50) else 0
        gain = ap2 - ap1
        signe = '+' if gain >= 0 else ''
        print(f'{ap2:>10.3f} {signe}{gain*100:>6.1f}%')
    else:
        print('(Phase 2 non exécutée)')

print('=' * 75)
print(f'  {"mAP@0.5 global":<20} {m1.box.map50:>10.3f} ', end='')
if results_p2_available:
    gain_map = m2.box.map50 - m1.box.map50
    signe = '+' if gain_map >= 0 else ''
    print(f'{m2.box.map50:>10.3f} {signe}{gain_map*100:>6.1f}%')
else:
    print()

# ── Graphique comparatif ──────────────────────────────────────────────────────
if results_p2_available:
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(CLASS_NAMES))
    w = 0.35

    b1 = ax.bar(x - w/2, m1.box.ap50, w, label='Phase 1 (freeze=10)', color='steelblue', alpha=0.85)
    b2 = ax.bar(x + w/2, m2.box.ap50, w, label='Phase 2 (full fine-tune)', color='coral',     alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([n.replace('_', '\n') for n in CLASS_NAMES], fontsize=9)
    ax.set_ylabel('AP@0.5')
    ax.set_ylim(0, 1)
    ax.set_title('Comparaison AP@0.5 par classe — Phase 1 vs Phase 2', fontsize=12, fontweight='bold')
    ax.axhline(y=0.7, color='gray', linestyle=':', linewidth=1, label='Seuil 0.70')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    for bar in b1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
    for bar in b2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig('./ppe_project/comparison_p1_p2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Graphique comparatif sauvegardé : ./ppe_project/comparison_p1_p2.png')

In [ ]:
# Section 7 (suite) — Conclusion et récapitulatif final

from pathlib import Path

BEST_P2 = Path('./ppe_project/runs/phase2_full_finetune/weights/best.pt')

print('═' * 65)
print('  RÉCAPITULATIF DU TP — TRANSFER LEARNING EPI HOSPITALIERS')
print('═' * 65)
print()
print('  Modèle de départ   : YOLOv8s (yolov8s.pt)')
print('  Pré-entraîné sur   : COCO (80 classes) + ImageNet backbone')
print('  Classes ajoutées   : 6 classes EPI hospitalières')
print()
print('  Phase 1 (freeze=10) :')
print(f'    mAP@0.5 = {m1.box.map50:.4f}  |  mAP@0.5:0.95 = {m1.box.map:.4f}')
print(f'    Précision = {m1.box.mp:.4f}  |  Recall = {m1.box.mr:.4f}')
print()

if BEST_P2.exists():
    gain = m2.box.map50 - m1.box.map50
    print('  Phase 2 (full fine-tune) :')
    print(f'    mAP@0.5 = {m2.box.map50:.4f}  |  mAP@0.5:0.95 = {m2.box.map:.4f}')
    print(f'    Précision = {m2.box.mp:.4f}  |  Recall = {m2.box.mr:.4f}')
    print(f'    Gain mAP@0.5 vs Phase 1 : +{gain*100:.1f}%')
    print()
    if gain > 0.05:
        print('  CONCLUSION : La Phase 2 a apporté une amélioration significative (> 5%).')
        print('  Le fine-tuning profond était justifié.')
    else:
        print('  CONCLUSION : La Phase 2 n\'a pas apporté d\'amélioration majeure.')
        print('  La Phase 1 (backbone gelé) était suffisante pour ce dataset.')
else:
    print('  Phase 2 non exécutée.')
    if m1.box.map50 >= 0.70:
        print('  CONCLUSION : La Phase 1 atteint mAP@0.5 ≥ 0.70 — résultats suffisants.')
    else:
        print('  CONCLUSION : Exécuter la Phase 2 (Section 6) pour améliorer les résultats.')

print()
print('═' * 65)
print('  Fichiers produits :')
for f in sorted(Path('./ppe_project').rglob('*.pt')):
    print(f'    {f}')
for f in sorted(Path('./ppe_project').glob('*.png')):
    print(f'    {f}')
print('═' * 65)